<a href="https://colab.research.google.com/github/coreprimejio/ev-server/blob/master-qa/PDF_Quiz_Generator_(Corrected).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import json
import os
import math
import uuid
import shutil
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Arc
from pylatex import Document, Command, Enumerate
from pylatex.utils import NoEscape

# --- UTILITY FUNCTION ---

def escape_and_format_latex(text):
    """Escapes special characters and formats specific symbols for LaTeX."""
    if not isinstance(text, str):
        return text
    # Basic LaTeX special characters
    text = text.replace('\\', r'\textbackslash{}')
    text = text.replace('&', r'\&')
    text = text.replace('%', r'\%')
    text = text.replace('$', r'\$')
    text = text.replace('#', r'\#')
    text = text.replace('_', r'\_')
    text = text.replace('{', r'\{')
    text = text.replace('}', r'\}')
    text = text.replace('~', r'\textasciitilde{}')
    text = text.replace('^', r'\textasciicircum{}')
    # Custom symbol replacements for math mode
    text = text.replace('☐', r' $\Box$ ')
    text = text.replace('∠', r'$\angle$')
    text = text.replace('Δ', r'$\Delta$')
    text = text.replace('⊥', r'$\perp$')
    text = text.replace('||', r'$\|$')
    return text

# --- FIGURE GENERATION FUNCTIONS ---

def create_angle_figure(angle_degrees, angle_type_label, points, output_filename):
    """Generates a figure of an angle with labels."""
    fig, ax = plt.subplots(figsize=(4, 3))
    vertex = (0, 0)
    arm_length = 1.0

    # --- **DEFINITIVE ARROW & LABEL FIX** ---
    # Using plt.quiver for reliable arrow drawing and a more robust label placement.

    # Arm 2 (rotated) vector
    angle_rad = np.deg2rad(angle_degrees)
    v1 = (arm_length * np.cos(angle_rad), arm_length * np.sin(angle_rad))

    # Arm 1 (horizontal) vector
    v2 = (arm_length, 0)

    # Draw the arms using quiver for consistency
    ax.quiver(vertex[0], vertex[1], v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='k', headwidth=5, headlength=7, width=0.005)
    ax.quiver(vertex[0], vertex[1], v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='k', headwidth=5, headlength=7, width=0.005)

    # New robust label placement logic
    if len(points) == 3:
        label_offset = 1.2 # Multiplier to place label just beyond the arrow tip
        # Top arm label (points[0])
        if points[0]:
            ax.text(v1[0] * label_offset, v1[1] * label_offset, f'${points[0]}$', fontsize=14, ha='center', va='center')
        # Vertex label (points[1])
        if points[1]:
            ax.text(vertex[0] - 0.15, vertex[1] - 0.15, f'${points[1]}$', fontsize=14, ha='center', va='center')
        # Bottom arm label (points[2])
        if points[2]:
            ax.text(v2[0] * label_offset, v2[1] - 0.05, f'${points[2]}$', fontsize=14, ha='center', va='center')

    # Draw the angle arc
    if angle_degrees == 90:
        ax.add_patch(Rectangle((vertex[0], vertex[1]), 0.15, 0.15, facecolor='none', edgecolor='k', lw=1))
    elif angle_degrees > 180:
        ax.add_patch(Arc(vertex, 0.6, 0.6, angle=0, theta1=angle_degrees, theta2=360, color='k', lw=1.5))
    else:
        ax.add_patch(Arc(vertex, 0.3, 0.3, angle=0, theta1=0, theta2=angle_degrees, color='k', lw=1.5))

    ax.set_xlim(-0.7, 1.7)
    ax.set_ylim(-0.7, 1.7)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')

    plt.savefig(output_filename, format='png', bbox_inches='tight', dpi=150)
    plt.close()

def create_polygon_figure(vertices, labels, output_filename):
    """Generates a figure of a polygon from a list of vertices and labels."""
    fig, ax = plt.subplots(figsize=(4, 4))

    polygon = Polygon(vertices, closed=True, facecolor='lightgrey', edgecolor='black', linewidth=2)
    ax.add_patch(polygon)

    if labels and len(labels) == len(vertices):
        for i, label in enumerate(labels):
            if label: # Only draw label if it's not an empty string
                x, y = vertices[i]
                offset_x = 0.15 if x > sum(v[0] for v in vertices) / len(vertices) else -0.15
                offset_y = 0.15 if y > sum(v[1] for v in vertices) / len(vertices) else -0.15
                ax.text(x + offset_x, y + offset_y, f'${label}$', fontsize=14, ha='center', va='center')

    all_x = [v[0] for v in vertices]
    all_y = [v[1] for v in vertices]
    ax.set_xlim(min(all_x) - 0.5, max(all_x) + 0.5)
    ax.set_ylim(min(all_y) - 0.5, max(all_y) + 0.5)

    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')

    plt.savefig(output_filename, format='png', bbox_inches='tight', dpi=150)
    plt.close()

# --- PDF GENERATION SCRIPT ---

def generate_quiz_pdf(quiz_data, pdf_filepath):
    """Reads a JSON object of questions and generates a single-column A4 PDF."""
    geometry_options = {"tmargin": "1in", "lmargin": "1in"}
    doc = Document(pdf_filepath, documentclass='article', document_options=['a4paper', '12pt'], geometry_options=geometry_options)

    doc.preamble.append(Command('usepackage', 'graphicx'))
    doc.preamble.append(Command('usepackage', 'amsmath'))
    doc.preamble.append(Command('usepackage', 'enumitem'))
    doc.preamble.append(Command('usepackage', 'xcolor'))
    doc.preamble.append(Command('usepackage', 'amssymb'))
    doc.preamble.append(Command('setlength', [NoEscape(r'\parindent'), '0pt']))
    doc.preamble.append(NoEscape(r'\definecolor{questioncolor}{RGB}{40,80,150}'))

    title = escape_and_format_latex(quiz_data.get('topic', 'Quiz'))
    class_level = escape_and_format_latex(quiz_data.get('class', ''))
    module_name = escape_and_format_latex(quiz_data.get('module', ''))

    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'{\Huge\bfseries ' + title + r'}\\[5pt]'))
    doc.append(NoEscape(r'{\Large\bfseries ' + module_name + r'}\\[10pt]'))
    doc.append(NoEscape(r'{\large ' + class_level + r'}'))
    doc.append(NoEscape(r'\end{center}\bigskip'))

    output_pdf_dir = os.path.dirname(pdf_filepath)
    figures_dir = os.path.join(output_pdf_dir, 'figures')
    os.makedirs(figures_dir, exist_ok=True)

    question_number = 1
    for q in quiz_data['questions']:
        try:
            # First, attempt to generate the figure. If this fails, the question is skipped.
            fig_full_path = None
            if 'figure_desc' in q:
                fig_desc = q['figure_desc']
                fig_name_only = f'q_{question_number}.png'
                fig_full_path = os.path.join(figures_dir, fig_name_only)
                fig_type = fig_desc.get('type_of_figure')

                if fig_type == 'angle':
                    create_angle_figure(fig_desc.get('angle_degrees', 45), fig_desc.get('angle_type_label'), fig_desc.get('points', []), fig_full_path)
                elif fig_type == 'polygon':
                    create_polygon_figure(fig_desc.get('vertices', []), fig_desc.get('labels', []), fig_full_path)

            # If figure generation is successful (or not needed), proceed to add question to PDF
            question_text = escape_and_format_latex(q['question'])
            doc.append(NoEscape(r'\textcolor{questioncolor}{\textbf{Question ' + str(question_number) + r':}} '))
            doc.append(NoEscape(question_text))
            doc.append(NoEscape(r'\par'))

            # Add the figure to the document if it exists
            if fig_full_path and os.path.exists(fig_full_path):
                fig_relative_path = os.path.join('figures', fig_name_only)
                doc.append(NoEscape(r'\begin{center}'))
                doc.append(NoEscape(r'\includegraphics[width=0.25\linewidth]{' + fig_relative_path.replace('\\', '/') + '}'))
                doc.append(NoEscape(r'\end{center}'))

            with doc.create(Enumerate(options=NoEscape(r'label=(\alph*)'))) as enum:
                for option in q['options']:
                    formatted_option = escape_and_format_latex(option)
                    enum.add_item(NoEscape(formatted_option))

            doc.append(NoEscape(r'\bigskip\hrule\bigskip'))

            question_number += 1
        except Exception:
            # Silently skip the question if any part of the process fails
            pass

    try:
        doc.generate_pdf(clean_tex=True)
        print(f"✅ Successfully generated PDF: {pdf_filepath}.pdf")
    except Exception as e:
        print(f"❌ PDF generation failed. Error: {e}")

if __name__ == '__main__':
    # --- User-defined paths ---
    input_dir = '/Users/ajaygupta/class5/fractions/json'
    output_dir = '/Users/ajaygupta/class5/fractions/pdfs'
    json_filename = 'fractions.json'

    # --- Script Execution ---
    if not os.path.exists(input_dir):
        os.makedirs(input_dir)
        print(f"📁 Created input directory: {input_dir}")
        print(f"Please place '{json_filename}' inside this directory.")

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"🧹 Cleaned up old output directory: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    json_file_path = os.path.join(input_dir, json_filename)

    pdf_filename = f"{os.path.splitext(json_filename)[0]}-{uuid.uuid4()}"
    pdf_full_path = os.path.join(output_dir, pdf_filename)

    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            quiz_json_object = json.load(f)
        generate_quiz_pdf(quiz_json_object, pdf_full_path)
    except FileNotFoundError:
        print(f"❌ Error: The file was not found at {json_file_path}")
    except json.JSONDecodeError as e:
        print(f"❌ Error: The file at {json_file_path} is not a valid JSON file. Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

🧹 Cleaned up old output directory: /Users/ajaygupta/class5/fractions/pdfs
✅ Successfully generated PDF: /Users/ajaygupta/class5/fractions/pdfs/fractions-87f7ec32-959e-47d5-960b-536a2faf9583.pdf
